In [ ]:
path="/data_lake/gold/intlpris/"

In [ ]:
from pyspark.sql import SparkSession
from impala.dbapi import connect
import pandas as pd
import os
import json
import requests
from pyspark.sql import functions as sf, types as T
import sys
from datetime import datetime, timedelta

spark = SparkSession.builder \
            .appName("Inteligência Prisional") \
            .config("spark.dynamicAllocation.enabled", "true") \
            .config("spark.dynamicAllocation.initialExecutors", "1") \
            .config("spark.dynamicAllocation.minExecutors", "1") \
            .config("spark.dynamicAllocation.maxExecutors", "4") \
            .config("spark.executor.memory", "4g") \
            .config("spark.executor.cores", "1") \
            .config("spark.driver.memory", "4g") \
            .config("spark.driver.cores", "1") \
            .config("spark.yarn.executor.memoryOverhead", "1g") \
            .config("spark.executor.memoryOverhead", "1g") \
            .config("spark.sql.parquet.int96RebaseModeInWrite", "LEGACY") \
            .config("spark.sql.parquet.datetimeRebaseModeInWrite", "LEGACY") \
            .master('local') \
            .enableHiveSupport() \
            .getOrCreate()

In [ ]:
def get_dtype_df(dataframe):
    string_cols = [col for col, dtype in dataframe.dtypes if dtype == "string"]
    
    # max em BYTES (UTF-8) para cada coluna string
    if string_cols:
        df_max_bytes = (
            dataframe.select([
                sf.max(sf.length(sf.encode(sf.col(col), "UTF-8"))).alias(col)
                for col in string_cols
            ])
            .first()
            .asDict()
        )
    else:
        df_max_bytes = {}
    
    df_schema = pd.json_normalize(json.loads(dataframe.schema.json()), record_path=['fields'])[['name', 'type']]

    def adjust_dtype(name, dtype):
        if dtype == "long":
            return "bigint"
        
        if dtype == 'string':
            max_length = df_max_bytes.get(name) or 1
            
            # ajusta para CHAR(n) se for até 16 caracteres, senão usa VARCHAR(n)
            if max_length <= 16:
                return f"char({max_length})"  # garantir CHAR(1) como mínimo
            else:
                return f"varchar({max_length})" 
            
        else:
            return dtype

    df_schema['type'] = df_schema.apply(lambda x: adjust_dtype(x['name'], x['type']), axis=1)
    return df_schema

def write_impala_table_partioned(df, impala_schema, impala_table, br_path):
    dtype_df = get_dtype_df(df)
    dtypes_sting = ",\n    ".join([f"{row['name']} {row['type']}" for index, row in dtype_df.iterrows()])
    creation_query = f"""CREATE EXTERNAL TABLE {impala_schema}.{impala_table} 
                        ({dtypes_sting})
                        STORED AS PARQUET LOCATION '{br_path}'
                        """
    drop_query = f"""DROP TABLE {impala_schema}.{impala_table} """
    conn = connect(host='worker03-prod.sejus.es.gov.br', 
                   port=21050, 
                   database='bronze', 
                   auth_mechanism='GSSAPI',
                   kerberos_service_name='impala',
                   use_ssl=True,
                   ca_cert="/var/lib/cloudera-scm-agent/agent-cert/cm-auto-global_cacerts.pem")

    cursor = conn.cursor()
    print(creation_query)
    print(drop_query)
    try:
        cursor.execute(drop_query)
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
    except: 
        cursor.execute(creation_query)
        cursor.execute(f"COMPUTE STATS {impala_schema}.{impala_table}")
        print(f"Estatísticas atualizadas para {impala_schema}.{impala_table}")
        
def enviar_gold_para_postgres(nome_tabela_origem, pk_postgres):
    """
    Cria/substitui uma tabela no PostgreSQL a partir de uma tabela Spark/Hive
    e define uma chave primária no campo informado, caso pk_postgres seja informado.

    Parâmetros:
        nome_tabela_origem : str
            Nome completo da tabela no Spark, ex: "gold.sinp_pres_loc_atual"

        pk_postgres : str
            Nome do campo que será chave primária no PostgreSQL.
            Se vier "" ou None, a tabela será criada sem PK.

    Premissas:
        - a tabela de origem já existe no Spark
        - o driver JDBC do PostgreSQL está disponível no cluster
        - o schema/tabela de destino no PostgreSQL terão o mesmo nome da origem
          Ex: gold.sinp_pres_loc_atual -> schema "sinp", tabela "sinp_pres_loc_atual"
    """

    url = "jdbc:postgresql://10.242.38.126:5432/sinp_db"
    usuario = "usr_sinp"
    senha = "u9oLzKOato#nksFZ"
    driver = "org.postgresql.Driver"

    partes = nome_tabela_origem.split(".")
    if len(partes) != 2:
        raise ValueError("Informe a tabela no formato schema.tabela. Ex: gold.sinp_pres_loc_atual")

    schema_destino = "sinp"
    tabela_destino = partes[1]

    spark.catalog.clearCache()
    spark.sql(f"REFRESH TABLE {nome_tabela_origem}")
    df = spark.table(nome_tabela_origem).cache()
    df.count()

    colunas_df = df.columns

    tem_pk = pk_postgres is not None and str(pk_postgres).strip() != ""
    pk_postgres = str(pk_postgres).strip() if pk_postgres is not None else ""

    if tem_pk and pk_postgres not in colunas_df:
        raise ValueError(f"A PK '{pk_postgres}' não existe na tabela de origem. Colunas disponíveis: {colunas_df}")

    def mapear_tipo_postgres(campo):
        tipo = campo.dataType.simpleString().lower()

        if tipo.startswith("string"):
            return "varchar"
        elif tipo.startswith("int"):
            return "integer"
        elif tipo.startswith("bigint") or tipo.startswith("long"):
            return "bigint"
        elif tipo.startswith("double"):
            return "double precision"
        elif tipo.startswith("float"):
            return "real"
        elif tipo.startswith("boolean"):
            return "boolean"
        elif tipo.startswith("timestamp"):
            return "timestamp"
        elif tipo.startswith("date"):
            return "date"
        elif tipo.startswith("smallint"):
            return "smallint"
        elif tipo.startswith("decimal"):
            return tipo.replace("decimal", "numeric")
        else:
            return "text"

    ddl_colunas = []
    for campo in df.schema.fields:
        nome_coluna = campo.name
        tipo_pg = mapear_tipo_postgres(campo)
        ddl_colunas.append(f'"{nome_coluna}" {tipo_pg}')

    ddl_create_schema = f'create schema if not exists "{schema_destino}"'
    ddl_drop_table = f'drop table if exists "{schema_destino}"."{tabela_destino}"'

    if tem_pk:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)},
                constraint pk_{tabela_destino} primary key ("{pk_postgres}")
            )
        '''
    else:
        ddl_create_table = f'''
            create table "{schema_destino}"."{tabela_destino}" (
                {", ".join(ddl_colunas)}
            )
        '''

    jvm = spark._sc._gateway.jvm
    jvm.java.lang.Class.forName(driver)

    conn = jvm.java.sql.DriverManager.getConnection(url, usuario, senha)
    stmt = conn.createStatement()

    try:
        stmt.execute(ddl_create_schema)
        stmt.execute(ddl_drop_table)
        stmt.execute(ddl_create_table)
    finally:
        stmt.close()
        conn.close()

    propriedades = {
        "user": usuario,
        "password": senha,
        "driver": driver
    }

    df.write \
        .mode("append") \
        .jdbc(
            url=url,
            table=f'{schema_destino}.{tabela_destino}',
            properties=propriedades
        )

    print(f"Tabela enviada com sucesso para o PostgreSQL: {schema_destino}.{tabela_destino}")
    if tem_pk:
        print(f"PK definida: {pk_postgres}")
    else:
        print("Tabela criada sem PK.")

In [ ]:
df_pres = spark.sql("""
with docs_base as (
    select
        cast(d.id_preso as string) as id_preso,
        d.id_documentotipo,
        regexp_replace(coalesce(d.presodocumento_numero, ''), '[^0-9]', '') as id_documento_limpo
    from bronze.infopen_preso_documentos d
    where d.presodocumento_numero is not null
      and regexp_replace(coalesce(d.presodocumento_numero, ''), '[^0-9]', '') <> ''
),
classificacao as (
    select
        id_preso,
        max(case when id_documentotipo in (18,19) then 1 else 0 end) as tem_doc_nacional,
        max(case when id_documentotipo = 26 then 1 else 0 end) as tem_passaporte
    from docs_base
    group by id_preso
),
filiacao as (
    select
        cast(id_preso as string) as id_preso,
        max(
            case
                when upper(regexp_replace(coalesce(presofiliacao_mae, ''), '\\s+', ' ')) rlike ' OU '
                    then trim(
                        regexp_extract(
                            regexp_replace(coalesce(presofiliacao_mae, ''), '\\s+', ' '),
                            '^(.*?)\\s+(?i:OU)\\s+.*$',
                            1
                        )
                    )
                else trim(regexp_replace(coalesce(presofiliacao_mae, ''), '\\s+', ' '))
            end
        ) as nome_mae,
        max(
            case
                when upper(regexp_replace(coalesce(presofiliacao_pai, ''), '\\s+', ' ')) rlike ' OU '
                    then trim(
                        regexp_extract(
                            regexp_replace(coalesce(presofiliacao_pai, ''), '\\s+', ' '),
                            '^(.*?)\\s+(?i:OU)\\s+.*$',
                            1
                        )
                    )
                else trim(regexp_replace(coalesce(presofiliacao_pai, ''), '\\s+', ' '))
            end
        ) as nome_pai
    from bronze.infopen_preso_filiacao
    group by cast(id_preso as string)
),
etnia_base as (
    select
        cast(cpe.id_preso as string) as id_preso,
        concat_ws(', ', collect_set(cor.corpeleetnia_descricao)) as etnia
    from bronze.infopen_preso_cor_pele_etnia cpe
    inner join bronze.infopen_cor_pele_etnia cor
        on cor.id_corpeleetnia = cpe.id_corpeleetnia
    group by cast(cpe.id_preso as string)
),
base_documento as (
    select
        d.id_preso,
        d.id_documentotipo,
        dt.documentotipo_descricao,
        d.id_documento_limpo,
        c.tem_doc_nacional,
        c.tem_passaporte,
        case
            when upper(regexp_replace(coalesce(p.preso_nome, ''), '\\s+', ' ')) rlike ' OU '
                then trim(
                    regexp_extract(
                        regexp_replace(coalesce(p.preso_nome, ''), '\\s+', ' '),
                        '^(.*?)\\s+(?i:OU)\\s+.*$',
                        1
                    )
                )
            else trim(regexp_replace(coalesce(p.preso_nome, ''), '\\s+', ' '))
        end as nome_pessoa,
        p.preso_sexo as sexo_pessoa,
        p.preso_datanascimento as data_nascimento_pessoa,
        p.preso_dataultimaprisao as data_ultima_prisao,
        f.nome_mae,
        f.nome_pai,
        e.etnia,
        row_number() over (
            partition by d.id_preso
            order by case
                when d.id_documentotipo = 19 then 1
                when d.id_documentotipo = 18 then 2
                when d.id_documentotipo = 26 then 3
                else 4
            end,
            length(d.id_documento_limpo) desc,
            d.id_documento_limpo desc
        ) as rn_doc
    from docs_base d
    inner join bronze.infopen_presos p
        on cast(p.id_preso as string) = d.id_preso
    inner join classificacao c
        on c.id_preso = d.id_preso
    left join bronze.infopen_documentos_tipos dt
        on dt.id_documentotipo = d.id_documentotipo
    left join filiacao f
        on f.id_preso = d.id_preso
    left join etnia_base e
        on e.id_preso = d.id_preso
),
melhor_por_preso as (
    select
        id_preso,
        id_documentotipo,
        documentotipo_descricao,
        id_documento_limpo,
        tem_doc_nacional,
        tem_passaporte,
        nome_pessoa,
        sexo_pessoa,
        data_nascimento_pessoa,
        data_ultima_prisao,
        nome_mae,
        nome_pai,
        etnia,
        case
            when tem_doc_nacional = 1 then concat('NAC_', id_documento_limpo)
            when tem_doc_nacional = 0 and tem_passaporte = 1 and id_documentotipo = 26 then concat('EST_', id_documento_limpo)
            else concat('NAC_', id_documento_limpo)
        end as id_pessoa
    from base_documento
    where rn_doc = 1
),
dedup_id_pessoa as (
    select
        *,
        row_number() over (
            partition by id_pessoa
            order by
                case when data_ultima_prisao is not null then 1 else 2 end,
                to_date(data_ultima_prisao) desc,
                case when data_nascimento_pessoa is not null then 1 else 2 end,
                to_date(data_nascimento_pessoa) desc,
                id_preso desc
        ) as rn_pessoa
    from melhor_por_preso
)
select
    cast(id_preso as string) as id_preso,
    id_pessoa,
    case
        when tem_doc_nacional = 1 then 'NACIONAL'
        when tem_doc_nacional = 0 and tem_passaporte = 1 then 'ESTRANGEIRO'
        else 'NACIONAL'
    end as origem,
    id_documentotipo as cod_documento_referencia,
    documentotipo_descricao as desc_documento_referencia,
    case
        when id_documentotipo = 19 then concat(
            substr(lpad(id_documento_limpo, 11, '0'), 1, 3), '.',
            substr(lpad(id_documento_limpo, 11, '0'), 4, 3), '.',
            substr(lpad(id_documento_limpo, 11, '0'), 7, 3), '-',
            substr(lpad(id_documento_limpo, 11, '0'), 10, 2)
        )
        else id_documento_limpo
    end as documento,
    nome_pessoa,
    sexo_pessoa,
    1 as flag_presidiario,
    to_date(data_nascimento_pessoa) as data_nascimento_pessoa,
    to_date(data_ultima_prisao) as data_ultima_prisao,
    nome_mae,
    nome_pai,
    etnia
from dedup_id_pessoa
where rn_pessoa = 1
""")

tabela="sinp_ent_pessoa"
df_pres.write.mode("overwrite").option("maxRecordsPerFile", 1_000_000).option("compression", "snappy").parquet(f"{path}{tabela}")
write_impala_table_partioned(df_pres, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_pessoa")

In [ ]:
#CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_CELA - APOS CORRIGIDO REMOVER

from pyspark.sql import functions as F

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_CELA.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_CELA_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_entrada_cela", F.col("dt_entrada_cela").cast("timestamp"))
    .withColumn("dt_registro", F.col("dt_registro").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)
df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_cela_corr")
spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_cela_corr
""")
spark.sql(f"""
CREATE TABLE bronze.infopen_preso_cela_corr
USING PARQUET
LOCATION '{destino}'
""")

# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM UP_CELA - APOS CORRIGIDO REMOVER

from pyspark.sql import functions as F

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_CELA.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_CELA_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_desativacao", F.col("dt_desativacao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_up_cela_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_up_cela_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_up_cela_corr
USING PARQUET
LOCATION '{destino}'
""")


path_origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_GALERIA.parquet"
path_destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/UP_GALERIA_CORR.parquet"

df_gal = (
    spark.read
    .parquet(path_origem)
    .withColumn(
        "dt_desativacao",
        F.to_timestamp(F.col("dt_desativacao").cast("string"), "yyyyMMdd")
    )
)

df_gal.write.mode("overwrite").parquet(path_destino)


spark.sql("drop table if exists bronze.infopen_up_galeria_corr")

spark.sql(f"""
create table bronze.infopen_up_galeria_corr
using parquet
location '{path_destino}'
""")

In [ ]:
spark.sql("""
create table if not exists bronze.sinp_estab_pris (
    sigla       string,
    unidade     string,
    municipio   string,
    endereco    string,
    cep         string,
    uf          string,
    pais        string,
    latitude    double,
    longitude   double
)
stored as parquet
location '/data_lake/bronze/intlpris/sinp_estab_pris'
""")

spark.sql("""
insert overwrite table bronze.sinp_estab_pris values
('CDPM' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE MARATAÍZES' ,'Marataízes' ,'Rua Espinha de Peixe, s/n, Bairro Acapulco' ,'29345-000' ,'ES','Brasil',-21.021596,-40.820643),
('CDPCI' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE CACHOEIRO DE ITAPEMIRIM' ,'Cachoeiro de Itapemirim' ,'Rodovia do Governador Lacerda de Aguiar, Km 01, Bairro Coronel Borges' ,'29306-095' ,'ES','Brasil',-20.8542956,-41.0902809),
('APACF' ,'ASSOCIAÇÃO PROTEÇÃO E ASSIST CONDENADOS CACH ITAPEMIRIM - CRS MASC' ,'Cachoeiro de Itapemirim' ,'Rodovia Cachoeiro-Monte Líbano, Village da Luz' ,'29309-500' ,'ES','Brasil',-20.82597,-41.11094),
('CPFCI' ,'CENTRO PRISIONAL FEMININO DE CACHOEIRO DE ITAPEMIRIM' ,'Cachoeiro de Itapemirim' ,'Fazenda Monte Líbano, s/n, Zona Rural' ,'29300-970' ,'ES','Brasil',-20.73284,-41.10819),
('PRCI' ,'PENITENCIÁRIA REGIONAL DE CACHOEIRO DE ITAPEMIRIM' ,'Cachoeiro de Itapemirim' ,'Fazenda Monte Líbano, s/n, Zona Rural' ,'29300-970' ,'ES','Brasil',-20.7327951,-41.10819),
('CDPG' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE GUARAPARI' ,'Guarapari' ,'Rodovia do Sol, Contorno Argilino Dario, Km 51,3, Maxinda' ,'29200-970' ,'ES','Brasil',-20.6777411,-40.509638),
('CDPFVV' ,'CENTRO DE DETENÇÃO PROVISÓRIA FEMININO DE VILA VELHA' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Xuri' ,'29100-000' ,'ES','Brasil',-20.4688193,-40.4644965),
('PEVV IV' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA IV' ,'Vila Velha' ,'Rodovia Governador Mario Covas, s/n, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694866,-40.4674836),
('PEVV V' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA V' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694866,-40.4675164),
('PEVV III' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA III' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694669,-40.4674585),
('PEVV VI' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA VI' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694669,-40.4675415),
('PEVV II' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA II' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694366,-40.4674528),
('PSVV' ,'PENITENCIÁRIA SEMIABERTA DE VILA VELHA' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694366,-40.4675472),
('CDPVV' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE VILA VELHA' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4694444,-40.4675),
('CDPVV 2' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE VILA VELHA II' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.4693995,-40.4675),
('PEVV I' ,'PENITENCIÁRIA ESTADUAL DE VILA VELHA I' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.46941,-40.4674692),
('PSVV II' ,'PENITENCIÁRIA SEMIABERTA DE VILA VELHA II' ,'Vila Velha' ,'Rodovia BR-101 Sul, Km 313, Fazenda Santa Fé, Xuri' ,'29100-000' ,'ES','Brasil',-20.46941,-40.4675308),
('CASCUVI' ,'CASA DE CUSTÓDIA DE VIANA' ,'Viana' ,'BR 262 KM 18 5 S N' ,'29130-010' ,'ES','Brasil',-20.382185,-40.4632381),
('CDPFV' ,'CENTRO DE DETENÇÃO PROVISÓRIA FEMININO DE VIANA' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821401,-40.4632381),
('CDPV II' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE VIANA II' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821472,-40.4632122),
('USP' ,'UNIDADE DE SAÚDE PRISIONAL' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821472,-40.463264),
('CTV' ,'CENTRO DE TRIAGEM DE VIANA' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821663,-40.4631945),
('DIMCME' ,'DIMCME - SERVIÇO PENAL' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821914,-40.4631907),
('PSME II' ,'PENITENCIÁRIA DE SEGURANÇA MÉDIA II' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821663,-40.4632817),
('EPEN' ,'ESCOLA PENITENCIÁRIA' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822144,-40.4632019),
('PSMA II' ,'PENITENCIÁRIA DE SEGURANÇA MÁXIMA II' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822144,-40.4632743),
('PAES' ,'PENITENCIÁRIA AGRÍCOLA DO ESPÍRITO SANTO' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822281,-40.4632246),
('PSMA I' ,'PENITENCIÁRIA DE SEGURANÇA MÁXIMA I' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3822281,-40.4632516),
('PSME I' ,'PENITENCIÁRIA DE SEGURANÇA MÉDIA I' ,'Viana' ,'Rodovia BR-262, Km 18,5' ,'29130-055' ,'ES','Brasil',-20.3821914,-40.4632855),
('CASCUVV' ,'CASA DE CUSTÓDIA DE VILA VELHA' ,'Vila Velha' ,'Rua Mestre Gomes, s/n, Pedra D’Água, Glória' ,'29122-100' ,'ES','Brasil',-20.3314703,-40.3137423),
('APAC' ,'ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA AOS CONDENADOS' ,'Vila Velha' ,'Praça Almirante Tamandaré, 193' ,'29.100-310' ,'ES','Brasil',-20.3321936,-40.300015),
('IRS' ,'INSTITUTO DE READAPTAÇÃO SOCIAL' ,'Vila Velha' ,'Rua Mestre Gomes, Glória' ,'29122-100' ,'ES','Brasil',-20.3261111,-40.3141667),
('PSC' ,'PENITENCIÁRIA SEMIABERTA DE CARIACICA' ,'Cariacica' ,'Rodovia Governador José Sette, s/n, Tucum' ,'29152-500' ,'ES','Brasil',-20.32244,-40.36908),
('CDPC' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE CARIACICA' ,'Cariacica' ,'Rua Mario Valentim, Santana' ,'29154-585' ,'ES','Brasil',-20.30972,-40.38562),
('QCGPMES' ,'QUARTEL DO COMANDO GERAL DA POLÍCIA MILITAR DO EST. DO ESPÍRITO SANTO' ,'Vitória' ,'Avenida Maruípe, 2111, São Cristóvão' ,'29048-463' ,'ES','Brasil',-20.2938889,-40.3122222),
('CPFC' ,'CENTRO PRISIONAL FEMININO DE CARIACICA' ,'Cariacica' ,'Rua Ofelino Meireles, Bairro Bubu' ,'29157-766' ,'ES','Brasil',-20.2836,-40.40944),
('PFC' ,'PENITENCIÁRIA FEMININA CARIACICA' ,'Cariacica' ,'Rua Armélio/Ofelino Meireles, Bubu' ,'29157-766' ,'ES','Brasil',-20.2835551,-40.40944),
('HCTP' ,'HOSPITAL DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO' ,'Cariacica' ,'Av. José Sette, s/n, Bairro Roças Velhas' ,'29156-970' ,'ES','Brasil',-20.27756,-40.40907),
('UCTP' ,'UNIDADE DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO' ,'Cariacica' ,'Av. José Sette, s/n, Bairro Roças Velhas' ,'29156-970' ,'ES','Brasil',-20.2776049,-40.40907),
('PEF' ,'PENITENCIÁRIA ESTADUAL FEMININA' ,'Cariacica' ,'Av. José Sette / região de Roças Velhas' ,'29156-970' ,'ES','Brasil',-20.2775151,-40.40907),
('CDPS' ,'CENTRO DE DETENÇÃO PROVISÓRIA DA SERRA' ,'Serra' ,'Rodovia do Contorno, BR-101, Km 278, Distrito de Queimados' ,'29160-000' ,'ES','Brasil',-20.1283333,-40.3077778),
('CDPA' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE ARACRUZ' ,'Aracruz' ,'Estrada Aracruz-Coqueiral, s/n, Fátima' ,'29192-205' ,'ES','Brasil',-19.9285265,-40.1542386),
('PRL' ,'PENITENCIÁRIA REGIONAL DE LINHARES' ,'Linhares' ,'Rua Projetada, s/n, Jardim Laguna' ,'29900-970' ,'ES','Brasil',-19.3768823,-40.0552718),
('CRL' ,'CENTRO DE RESSOCIALIZAÇÃO DE LINHARES' ,'Linhares' ,'Rodovia ES-440, Km 02, Bebedouro' ,'29900-970' ,'ES','Brasil',-19.4059,-40.0499),
('CDPCOL' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.5498025,-40.6272813),
('CPFCOL' ,'CENTRO PRISIONAL FEMININO DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.5497576,-40.6272813),
('PRCOL' ,'PENITENCIÁRIA REGIONAL DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.549825,-40.62724),
('PSMECOL' ,'PENITENCIÁRIA DE SEGURANÇA MÉDIA DE COLATINA' ,'Colatina' ,'Córrego Santa Fé, s/n' ,'29700-970' ,'ES','Brasil',-19.549825,-40.6273226),
('PSMCOL' ,'PENITENCIÁRIA SEMIABERTA MASCULINA DE COLATINA' ,'Colatina' ,'Avenida das Nações, s/n, Bairro Benjamin Carlos dos Santos (IBC)' ,'29712-408' ,'ES','Brasil',-19.517611,-40.61206),
('CDPSDN' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE SÃO DOMINGOS DO NORTE' ,'São Domingos do Norte' ,'Córrego Braço do Sul, Km 80, s/n' ,'29745-000' ,'ES','Brasil',-19.0169,-40.5358),
('PRBSF' ,'PENITENCIÁRIA REGIONAL DE BARRA DE SÃO FRANCISCO' ,'Barra de São Francisco' ,'Rodovia ES-320, Km 02' ,'29800-000' ,'ES','Brasil',-18.7525,-40.893),
('PSSM' ,'PENITENCIÁRIA SEMIABERTA DE SÃO MATEUS' ,'São Mateus' ,'Rodovia Governador Mario Covas, BR-101 Norte, Km 72,5, s/n, Rio Preto da Rodovia' ,'29940-800' ,'ES','Brasil',-18.7167449,-39.8594),
('CDPSM' ,'CENTRO DE DETENÇÃO PROVISÓRIA DE SÃO MATEUS' ,'São Mateus' ,'BR-101 Norte, Km 72,5, Fazenda Rancho das Telhas, Zona Rural' ,'29930-000' ,'ES','Brasil',-18.7167,-39.8594),
('PRSM' ,'PENITENCIÁRIA REGIONAL DE SÃO MATEUS' ,'São Mateus' ,'Rodovia Governador Mario Covas, BR-101 Norte, Km 72,5, s/n, Rio Preto da Rodovia' ,'29940-800' ,'ES','Brasil',-18.7166551,-39.8594);
""")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# BASE DE LOCALIZAÇÃO DO PRESO
# ADAPTADA PARA A NOVA VISÃO:
#   - CELA + GALERIA + ESTABELECIMENTO
# SEM PERDER A LÓGICA JÁ ESTABELECIDA
# ============================================================

df_base_loc = spark.sql("""
with loc as (
    select
        e.id_estabelecimento,
        c.id_galeria,
        c.id_cela,

        c.nm_cela,
        c.id_tipo_cela,
        c.qt_capacidadeprojetada,
        c.qt_capacidadeadaptada,
        c.qt_metros_cela,
        c.observacao_cela,
        c.sexo_cela,
        c.dt_desativacao as dt_desativacao_cela,

        g.descricao_galeria,
        g.dt_desativacao as dt_desativacao_galeria,

        e.id_orgao,
        e.id_estabelecimentotipo,
        e.id_estabelecimentotiposeguranca,
        e.estabelecimento_nome,
        e.estabelecimento_sigla,
        e.estabelecimento_cnpj,
        e.estabelecimento_email,
        e.estabelecimento_capacidade,
        e.estabelecimento_macrorregiao,
        e.estabelecimento_situacaoregistro,

        row_number() over (
            partition by c.id_estabelecimento, c.id_galeria, c.id_cela
            order by
                c.dt_desativacao desc nulls last,
                g.dt_desativacao desc nulls last,
                c.nm_cela
        ) as rn_cela
    from bronze.infopen_estabelecimentos e
    left join 
    bronze.infopen_up_cela_corr c on c.id_estabelecimento = e.id_estabelecimento
    left join bronze.infopen_up_galeria_corr g
        on c.id_galeria = g.id_galeria
       and c.id_estabelecimento = g.id_estabelecimento
       ),

base as (
    select
        c.id_presocela,

        l.id_estabelecimento,
        cast(c.id_preso as string) as id_preso,
        p.id_pessoa,

        l.id_galeria,
        l.id_cela,

        l.nm_cela,
        l.id_tipo_cela,
        l.qt_capacidadeprojetada,
        l.qt_capacidadeadaptada,
        l.qt_metros_cela,
        l.observacao_cela,
        l.sexo_cela,
        l.dt_desativacao_cela,

        l.descricao_galeria,
        l.dt_desativacao_galeria,

        l.id_orgao,
        l.id_estabelecimentotipo,
        l.id_estabelecimentotiposeguranca,
        l.estabelecimento_nome,
        l.estabelecimento_sigla,
        l.estabelecimento_cnpj,
        l.estabelecimento_email,
        l.estabelecimento_capacidade,
        l.estabelecimento_macrorregiao,
        l.estabelecimento_situacaoregistro,

        c.st_cela_ativa,
        c.dt_entrada_cela,
        c.id_usuario_registro,
        c.dt_registro,
        c.observacao_presocela,

        r.municipio,
        r.endereco,
        r.cep,
        r.uf,
        r.pais,
        r.latitude,
        r.longitude,

        row_number() over (
            partition by coalesce(
                cast(c.id_preso as string),
                concat(
                    'SEM_PRESO_',
                    cast(l.id_estabelecimento as string), '_',
                    coalesce(cast(l.id_galeria as string), '0'), '_',
                    coalesce(cast(l.id_cela as string), '0')
                )
            )
            order by
                c.dt_entrada_cela desc nulls last,
                c.dt_registro desc nulls last,
                c.id_presocela desc nulls last
        ) as rn
    from bronze.infopen_estabelecimentos e
    left join loc l
        on l.id_estabelecimento = e.id_estabelecimento
       and l.rn_cela = 1
    left join tmp_preso_cela_corr c
        on c.id_estabelecimento = l.id_estabelecimento
       and c.id_galeria = l.id_galeria
       and c.id_cela = l.id_cela
    left join gold.sinp_ent_pessoa p
        on cast(c.id_preso as string) = cast(p.id_preso as string)
    left join bronze.sinp_estab_pris r
        on upper(trim(e.estabelecimento_sigla)) = upper(trim(r.sigla))

)

select *
from base
""")

# ============================================================
# ENTIDADE: ESTABELECIMENTO
# ============================================================

df_ent_estabelecimento = spark.sql("""
    SELECT 
        cast(e.id_estabelecimento as string) as id_estabelecimento,
        cast(e.id_orgao as string) as id_orgao,
        cast(e.id_estabelecimentotipo as string) as id_estabelecimentotipo,
        case
            when upper(trim(e.estabelecimento_nome)) like '%HOSPITAL DE CUSTÓDIA%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%TRATAMENTO PSIQUIÁTRICO%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE SAÚDE PRISIONAL%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO%' then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when upper(trim(e.estabelecimento_nome)) like '%ESCOLA PENITENCIÁRIA%' then 'UNIDADE ESPECIAL / APOIO / FORMAÇÃO'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA AOS CONDENADOS%' then 'APAC / ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO PROTEÇÃO E ASSIST CONDENADOS%' then 'APAC / ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA'
            when upper(trim(e.estabelecimento_nome)) like '%INSTITUTO DE READAPTAÇÃO SOCIAL%' then 'READAPTAÇÃO / RESSOCIALIZAÇÃO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE RESSOCIALIZAÇÃO%' then 'READAPTAÇÃO / RESSOCIALIZAÇÃO'
            when upper(trim(e.estabelecimento_nome)) like '%CASA DE CUSTÓDIA%' then 'CASA DE CUSTÓDIA'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE TRIAGEM%' then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE DETENÇÃO PROVISÓRIA%' then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO PRISIONAL%' then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when upper(trim(e.estabelecimento_nome)) like '%QUARTEL DO COMANDO GERAL%' then 'UNIDADE MILITAR / POLICIAL'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA AGRÍCOLA%' then 'PENITENCIÁRIA AGRÍCOLA'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA%' then 'PENITENCIÁRIA'
            when upper(trim(e.estabelecimento_nome)) like '%SERVIÇO PENAL%' then 'SERVIÇO PENAL / ADMINISTRATIVO'
            when e.id_estabelecimentotipo = 1  then 'SERVIÇO PENAL / ADMINISTRATIVO'
            when e.id_estabelecimentotipo = 4  then 'PENITENCIÁRIA'
            when e.id_estabelecimentotipo = 5  then 'PENITENCIÁRIA AGRÍCOLA'
            when e.id_estabelecimentotipo = 6  then 'UNIDADE DE SAÚDE / CUSTÓDIA PSIQUIÁTRICA'
            when e.id_estabelecimentotipo = 8  then 'UNIDADE ESPECIAL / APOIO / FORMAÇÃO / APAC'
            when e.id_estabelecimentotipo = 12 then 'READAPTAÇÃO / RESSOCIALIZAÇÃO'
            when e.id_estabelecimentotipo = 14 then 'CENTRO DE DETENÇÃO / TRIAGEM / CENTRO PRISIONAL'
            when e.id_estabelecimentotipo = 15 then 'CASA DE CUSTÓDIA'
            when e.id_estabelecimentotipo = 17 then 'UNIDADE MILITAR / POLICIAL'
            else 'NÃO CLASSIFICADO'
        end as ds_estabelecimentotipo,
        cast(e.id_estabelecimentotiposeguranca as string) as id_estabelecimentotiposeguranca,
        case
            when upper(trim(e.estabelecimento_nome)) like '%SEMIABERTA%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%APAC%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO DE PROTEÇÃO E ASSISTÊNCIA AOS CONDENADOS%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%ASSOCIAÇÃO PROTEÇÃO E ASSIST CONDENADOS%' then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANÇA MÉDIA%' then 'SEGURANÇA MÉDIA'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANCA MEDIA%' then 'SEGURANÇA MÉDIA'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANÇA MÁXIMA%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%SEGURANCA MAXIMA%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA I%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA II%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA III%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%VILA VELHA VI%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%QUARTEL DO COMANDO GERAL%' then 'SEGURANÇA MÁXIMA'
            when upper(trim(e.estabelecimento_nome)) like '%HOSPITAL DE CUSTÓDIA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%TRATAMENTO PSIQUIÁTRICO%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE SAÚDE PRISIONAL%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%UNIDADE DE CUSTÓDIA E TRATAMENTO PSIQUIÁTRICO%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE DETENÇÃO PROVISÓRIA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO DE TRIAGEM%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CENTRO PRISIONAL%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%CASA DE CUSTÓDIA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA REGIONAL%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA ESTADUAL FEMININA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA FEMININA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when upper(trim(e.estabelecimento_nome)) like '%PENITENCIÁRIA AGRÍCOLA%' then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            when e.id_estabelecimentotiposeguranca = 1 then 'SEMIABERTO / BAIXA CONTENÇÃO / ALTERNATIVO'
            when e.id_estabelecimentotiposeguranca = 2 then 'SEGURANÇA MÉDIA'
            when e.id_estabelecimentotiposeguranca = 3 then 'SEGURANÇA MÁXIMA'
            when e.id_estabelecimentotiposeguranca = 4 then 'PADRÃO / OUTROS / NÃO ESPECIALIZADO'
            else 'NÃO CLASSIFICADO'
        end as ds_estabelecimentotiposeguranca,
        e.estabelecimento_nome,
        e.estabelecimento_sigla,
        e.estabelecimento_cnpj,
        e.estabelecimento_email,
        e.estabelecimento_capacidade,
        e.estabelecimento_macrorregiao,
        e.estabelecimento_situacaoregistro,
        p.municipio,
        p.endereco,
        p.cep,
        p.uf,
        p.pais,
        p.latitude,
        p.longitude
    FROM bronze.infopen_estabelecimentos e
    inner join bronze.sinp_estab_pris p
        on upper(trim(e.estabelecimento_sigla)) = upper(trim(p.sigla))
""")

# ============================================================
# ENTIDADE: GALERIA
# ============================================================

df_ent_galeria = (
    df_base_loc
    .filter(F.col("id_galeria").isNotNull())
    .select(
        F.md5(
            F.concat(
                F.lit("E"),
                F.coalesce(F.col("id_estabelecimento").cast("string"), F.lit("")),
                F.lit("G"),
                F.coalesce(F.col("id_galeria").cast("string"), F.lit(""))
            )
        ).alias("id_galeria"),

        F.col("id_estabelecimento").cast("string").alias("id_estabelecimento"),
        F.col("id_galeria").cast("string").alias("id_galeria_original"),

        F.when(
            F.col("descricao_galeria").isNotNull() & (F.length(F.trim(F.col("descricao_galeria"))) > 0),
            F.trim(F.col("descricao_galeria"))
        ).otherwise(
            F.concat(
                F.lit("Galeria "),
                F.when(
                    F.col("id_galeria").cast("string").rlike("^[0-9]+$"),
                    F.lpad(F.col("id_galeria").cast("string"), 3, "0")
                ).otherwise(F.col("id_galeria").cast("string"))
            )
        ).alias("nome_galeria"),

        F.trim(F.col("descricao_galeria")).alias("descricao_galeria"),
        F.col("dt_desativacao_galeria").alias("dt_desativacao_galeria")
    )
    .distinct()
)

# ============================================================
# ENTIDADE: CELA
# ============================================================

df_ent_cela = (
    df_base_loc
    .filter(
        F.col("id_galeria").isNotNull() &
        F.col("id_cela").isNotNull()
    )
    .select(
        F.md5(
            F.concat(
                F.lit("E"),
                F.coalesce(F.col("id_estabelecimento").cast("string"), F.lit("")),
                F.lit("G"),
                F.coalesce(F.col("id_galeria").cast("string"), F.lit("")),
                F.lit("C"),
                F.coalesce(F.col("id_cela").cast("string"), F.lit(""))
            )
        ).alias("id_cela"),

        F.md5(
            F.concat(
                F.lit("E"),
                F.coalesce(F.col("id_estabelecimento").cast("string"), F.lit("")),
                F.lit("G"),
                F.coalesce(F.col("id_galeria").cast("string"), F.lit(""))
            )
        ).alias("id_galeria"),

        F.col("id_estabelecimento").cast("string").alias("id_estabelecimento_origem"),
        F.col("id_galeria").cast("string").alias("id_galeria_origem"),
        F.col("id_cela").cast("string").alias("id_cela_origem"),

        F.when(
            F.col("nm_cela").isNotNull() & (F.length(F.trim(F.col("nm_cela"))) > 0),
            F.trim(F.col("nm_cela"))
        ).otherwise(
            F.concat(
                F.lit("Cela "),
                F.when(
                    F.col("id_cela").cast("string").rlike("^[0-9]+$"),
                    F.lpad(F.col("id_cela").cast("string"), 3, "0")
                ).otherwise(F.col("id_cela").cast("string"))
            )
        ).alias("nome_cela"),

        F.col("id_tipo_cela").cast("int").alias("id_tipo_cela"),
        F.col("qt_capacidadeprojetada").cast("int").alias("qt_capacidade_projetada"),
        F.col("qt_capacidadeadaptada").cast("int").alias("qt_capacidade_adaptada"),
        F.col("qt_metros_cela").cast("int").alias("qt_metros_cela"),
        F.trim(F.col("observacao_cela")).alias("observacao_cela"),
        F.col("sexo_cela").alias("sexo_cela"),
        F.col("dt_desativacao_cela").alias("dt_desativacao_cela"),
        F.trim(F.col("descricao_galeria")).alias("descricao_galeria"),
        F.col("estabelecimento_nome").alias("estabelecimento_nome"),

        F.concat_ws(
            " > ",
            F.col("estabelecimento_nome"),
            F.col("descricao_galeria"),
            F.when(
                F.col("nm_cela").isNotNull() & (F.length(F.trim(F.col("nm_cela"))) > 0),
                F.trim(F.col("nm_cela"))
            ).otherwise(
                F.concat(
                    F.lit("Cela "),
                    F.when(
                        F.col("id_cela").cast("string").rlike("^[0-9]+$"),
                        F.lpad(F.col("id_cela").cast("string"), 3, "0")
                    ).otherwise(F.col("id_cela").cast("string"))
                )
            )
        ).alias("ds_localizacao_completa")
    )
    .dropDuplicates(["id_cela"])
)

In [ ]:
# ============================================================
# ESCRITA GOLD
# ============================================================

tabela = "sinp_ent_estabelecimento"
df_ent_estabelecimento.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ent_estabelecimento, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_estabelecimento")

tabela = "sinp_ent_galeria"
df_ent_galeria.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ent_galeria, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_galeria")

tabela = "sinp_ent_cela"
df_ent_cela.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ent_cela, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_cela")

In [ ]:
rel_passo = spark.sql("""
select
        id as id_mov,
        trim(cast(infopen as string)) as id_preso,
        nome as nome_livro,
        trim(cast(cela_atual as string)) as cela_atual,
        trim(cast(cela_destino as string)) as cela_destino,
        autorizacao,
        motivo,
        cast(data_registro as timestamp) as dt_movimentacao,
        equipe_id,
        presidio_id
    from bronze.livros_acesso_unidade_trocacela
    where trim(cast(infopen as string)) is not null
      and trim(cast(infopen as string)) <> ''
      and cast(data_registro as timestamp) is not null
      and (
            (trim(cast(cela_atual as string)) is not null and trim(cast(cela_atual as string)) <> '')
         or (trim(cast(cela_destino as string)) is not null and trim(cast(cela_destino as string)) <> '')
      )

""")

tabela = "sinp_preso_cela_p1"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")

#rel_passo.show(30,False)

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_preso_cela_p1")

rel_passo = spark.sql("""
select
        s.id_mov as id_evento_origem,
        id_preso,
        s.cela_atual as id_cela_origem_livro,
        s.dt_movimentacao as dt_entrada_uso_cela,
        s.dt_movimentacao as dt_evento_referencia,
        'PRIMEIRA_ORIGEM_INFERIDA' as origem_regra,
        1 as prioridade_regra,
        s.autorizacao as autorizacao,
        s.motivo as motivo,
        s.nome_livro as nome_livro,
        s.equipe_id as equipe_id,
        s.presidio_id as presidio_id
    from (
        select
            id_preso,
            min(
                named_struct(
                    'dt_movimentacao', dt_movimentacao,
                    'id_mov', id_mov,
                    'cela_atual', cela_atual,
                    'autorizacao', autorizacao,
                    'motivo', motivo,
                    'nome_livro', nome_livro,
                    'equipe_id', equipe_id,
                    'presidio_id', presidio_id
                )
            ) as s
        from gold.sinp_preso_cela_p1
        where cela_atual is not null
          and cela_atual <> ''
        group by id_preso
    ) a
""")

tabela = "sinp_preso_cela_p2"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")

#rel_passo.show(30,False)

rel_passo = spark.sql("""
    select
        id_mov as id_evento_origem,
        id_preso,
        cela_destino as id_cela_origem_livro,
        dt_movimentacao as dt_entrada_uso_cela,
        dt_movimentacao as dt_evento_referencia,
        'DESTINO_MOVIMENTACAO' as origem_regra,
        2 as prioridade_regra,
        autorizacao,
        motivo,
        nome_livro,
        equipe_id,
        presidio_id
    from gold.sinp_preso_cela_p1
    where cela_destino is not null
      and cela_destino <> ''
""")

tabela = "sinp_preso_cela_p3"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")


spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_preso_cela_p2")
spark.sql("refresh table gold.sinp_preso_cela_p3")


#rel_passo.show(30,False)

rel_passo = spark.sql("""
    select * from gold.sinp_preso_cela_p2
    union all
    select * from gold.sinp_preso_cela_p3
""")

tabela = "sinp_preso_cela_p4"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")
    



In [ ]:
spark.sql("drop table if exists gold.sinp_preso_cela_p1")
spark.sql("drop table if exists gold.sinp_preso_cela_p2")
spark.sql("drop table if exists gold.sinp_preso_cela_p3")

spark.catalog.clearCache()
spark.sql("refresh table gold.sinp_preso_cela_p4")

rel_passo = spark.sql("""
select
    md5(concat_ws('||',
        trim(cast(id_preso as string)),
        cast(dt_entrada_uso_cela as string),
        trim(cast(id_cela_origem_livro as string))
    )) as id_preso_data_cela,
    id_evento_origem as id_mov,
    id_preso,
    id_cela_origem_livro,
    dt_entrada_uso_cela,
    dt_evento_referencia,
    origem_regra,
    prioridade_regra,
    autorizacao,
    motivo,
    nome_livro,
    equipe_id,
    presidio_id,
    row_number() over (
        partition by md5(concat_ws('||',
            trim(cast(id_preso as string)),
            cast(dt_entrada_uso_cela as string),
            trim(cast(id_cela_origem_livro as string))
        ))
        order by
            prioridade_regra asc,
            dt_evento_referencia asc,
            origem_regra asc
    ) as rn
from gold.sinp_preso_cela_p4
where id_preso is not null
  and trim(cast(id_preso as string)) <> ''
  and dt_entrada_uso_cela is not null
  and id_cela_origem_livro is not null
  and trim(cast(id_cela_origem_livro as string)) <> ''
""")

#rel_passo.show(30,False)

tabela = "sinp_preso_cela_p5"
rel_passo.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(rel_passo, "gold", tabela, f"{path}{tabela}")

In [ ]:
df_ultima_data = spark.sql("""
select id_preso, max(dt_entrada_uso_cela) as dataRef from gold.sinp_preso_cela_p5 group by 1
""")
tabela = "ref_maxData"
df_ultima_data.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_ultima_data, "gold", tabela, f"{path}{tabela}")




df_base_preso_cela = spark.sql("""
select
    p5.id_preso_data_cela as id_rel_preso_cela,
    c.id_cela,
    p.id_pessoa,
    p5.id_mov,
    p5.id_preso,
    p5.id_cela_origem_livro as numero_cela,
    p5.dt_entrada_uso_cela,
    p5.dt_evento_referencia,
    p5.origem_regra,
    p5.prioridade_regra,
    p5.autorizacao,
    p5.motivo,
    p5.nome_livro,
    p5.equipe_id,
    p5.presidio_id,
    case when m.dataRef is null then 'Anterior' else 'Atual' end as situacao
from gold.sinp_preso_cela_p5 p5
    inner join gold.sinp_ent_cela c on p5.id_cela_origem_livro = c.id_cela_origem
    inner join gold.sinp_ent_pessoa p on p5.id_preso = p.id_preso
    left join gold.ref_maxData m on p5.id_preso = m.id_preso and p5.dt_entrada_uso_cela = m.dataRef
where rn= 1
""")

tabela = "sinp_rel_preso_cela"
df_base_preso_cela.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_preso_cela, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rel_preso_cela")


df_base_preso_cela_exc = spark.sql("""
select
    p5.id_preso_data_cela as id_rel_preso_cela,
    c.id_cela,
    p.id_pessoa,
    p5.id_mov,
    p5.id_preso,
    p5.id_cela_origem_livro as numero_cela,
    p5.dt_entrada_uso_cela,
    p5.dt_evento_referencia,
    p5.origem_regra,
    p5.prioridade_regra,
    p5.autorizacao,
    p5.motivo,
    p5.nome_livro,
    p5.equipe_id,
    p5.presidio_id,
    case when m.dataRef is null then 'Anterior' else 'Atual' end as situacao
from gold.sinp_preso_cela_p5 p5
    inner join gold.sinp_ent_cela c on p5.id_cela_origem_livro = c.id_cela_origem
    inner join gold.sinp_ent_pessoa p on p5.id_preso = p.id_preso
    left join gold.ref_maxData m on p5.id_preso = m.id_preso and p5.dt_entrada_uso_cela = m.dataRef
where rn> 1
""")

tabela = "sinp_rel_preso_cela_exc"
df_base_preso_cela_exc.write.mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")
write_impala_table_partioned(df_base_preso_cela_exc, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_rel_preso_cela")


In [ ]:
# ============================================================
# CORREÇÃO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_SOCIAL
# APÓS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_SOCIAL.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_SOCIAL_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_social_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_social_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_social_corr
USING PARQUET
LOCATION '{destino}'
""")

# ============================================================
# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_PSICO
# APOS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PSICO.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PSICO_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
    .withColumn("dt_ultima_alteracao", F.col("dt_ultima_alteracao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_psico_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_psico_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_psico_corr
USING PARQUET
LOCATION '{destino}'
""")

# ============================================================
# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_CRIMINAL
# APOS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_CRIMINAL.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_CRIMINAL_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
    .withColumn("dt_ultima_alteracao", F.col("dt_ultima_alteracao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_criminal_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_criminal_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_criminal_corr
USING PARQUET
LOCATION '{destino}'
""")


# ============================================================
# CORRECAO DEVIDO ERRO DE TIPO DE DADO EM PRESO_PRONTUARIO_PROFISSIONAL
# APOS CORRIGIDO REMOVER
# ============================================================

origem = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PROFISSIONAL.parquet"
destino = "hdfs://hahdfsprod/data_lake/bronze/INFOPEN/PRESO_PRONTUARIO_PROFISSIONAL_CORR.parquet"

df_corr = (
    spark.read.parquet(origem)
    .withColumn("dt_cadastro_ficha", F.col("dt_cadastro_ficha").cast("timestamp"))
    .withColumn("dt_ultima_alteracao", F.col("dt_ultima_alteracao").cast("timestamp"))
)

df_corr.write.mode("overwrite").parquet(destino)

df_valid = spark.read.parquet(destino)
df_valid.createOrReplaceTempView("tmp_preso_prontuario_profissional_corr")

spark.sql("""
DROP TABLE IF EXISTS bronze.infopen_preso_prontuario_profissional_corr
""")

spark.sql(f"""
CREATE TABLE bronze.infopen_preso_prontuario_profissional_corr
USING PARQUET
LOCATION '{destino}'
""")

In [ ]:
prontsoc = spark.sql("""
SELECT
    md5(concat(cast(ps.id_prontuario_social as string), cast(p.id_pessoa as string))) as id_pessoa_prontsoc,
    p.id_pessoa,
    ps.*
FROM bronze.infopen_preso_prontuario_social_corr ps
inner join gold.sinp_ent_pessoa p
    on cast(ps.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_soc"
prontsoc.write.mode("overwrite").option("maxRecordsPerFile", 1_000_000).option("compression", "snappy").parquet(f"{path}{tabela}")
write_impala_table_partioned(prontsoc, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_soc", "id_pessoa_prontsoc")

tabela = "sinp_fil_pront_soc"



# ============================================================
# PRONTUARIO CRIMINAL
# ============================================================

prontcrim = spark.sql("""
SELECT
    md5(concat(cast(pc.id_prontuario_criminal as string), cast(p.id_pessoa as string))) as id_pessoa_prontcrim,
    p.id_pessoa,
    pc.*
FROM bronze.infopen_preso_prontuario_criminal_corr pc
inner join gold.sinp_ent_pessoa p
    on cast(pc.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_crim"

prontcrim.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(prontcrim, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_crim", "id_pessoa_prontcrim")


# ============================================================
# PRONTUARIO PROFISSIONAL
# ============================================================

prontprof = spark.sql("""
SELECT
    md5(concat(cast(pp.id_prontuario_profissional as string), cast(p.id_pessoa as string))) as id_pessoa_prontprof,
    p.id_pessoa,
    pp.*
FROM bronze.infopen_preso_prontuario_profissional_corr pp
inner join gold.sinp_ent_pessoa p
    on cast(pp.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_prof"

prontprof.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(prontprof, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_prof", "id_pessoa_prontprof")

# ============================================================
# PRONTUARIO PSICO
# ============================================================

prontpsico = spark.sql("""
SELECT
    md5(concat(cast(pps.id_prontuario_psico as string), cast(p.id_pessoa as string))) as id_pessoa_prontpsico,
    p.id_pessoa,
    pps.*
FROM bronze.infopen_preso_prontuario_psico_corr pps
inner join gold.sinp_ent_pessoa p
    on cast(pps.id_preso as string) = cast(p.id_preso as string)
""")

tabela = "sinp_fil_pront_psico"

prontpsico.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(prontpsico, "gold", tabela, f"{path}{tabela}")

enviar_gold_para_postgres("gold.sinp_fil_pront_psico", "id_pessoa_prontpsico")

In [ ]:
df_alvaras = spark.sql("""
with alvaras as (
    select
        *,
        cast(id_preso as string) as id_preso_str
    from bronze.infopen_vw_alvaras
)
select
    a.*,
    p.id_pessoa,b.beneficio_nome,b.beneficio_descricao
from alvaras a
left join gold.sinp_ent_pessoa p
    on a.id_preso_str = cast(p.id_preso as string)
inner join bronze.infopen_minio_beneficios b on a.id_beneficio = b.id_beneficio
""")

tabela = "sinp_ent_alvaras"
df_alvaras.write.mode("overwrite").option("maxRecordsPerFile", 1_000_000).option("compression", "snappy").parquet(f"{path}{tabela}")
write_impala_table_partioned(df_alvaras, "gold", tabela, f"{path}{tabela}")



enviar_gold_para_postgres("gold.sinp_ent_alvaras", "id_alvara")

In [ ]:
from pyspark.sql import functions as F, types as T, Row
import hashlib
from datetime import datetime, timedelta, date

# ============================================================
# FUNCOES AUXILIARES
# ============================================================

def _hash(*vals):
    txt = "||".join("" if v is None else str(v) for v in vals)
    return hashlib.md5(txt.encode("utf-8")).hexdigest()

def _to_python_dt(v):
    if v is None:
        return None
    return v

def _today_midnight():
    now = datetime.now()
    return datetime(now.year, now.month, now.day)

# ============================================================
# 1. BASE UNIFICADA DE EVENTOS
# ============================================================

df_eventos_macro = spark.sql("""
select
    cast(id_preso as string) as id_preso,
    cast(id_pessoa as string) as id_pessoa,
    nome_pessoa,
    cast(id_movimentacao as string) as id_movimentacao,
    cast(id_tipomovimentacao as string) as id_tipomovimentacao,
    ds_tipo_mov,
    cast(movimentacao_data as timestamp) as dt_evento,
    cast(id_estabelecimentosecurity as string) as id_estabelecimento,
    observacao,
    categoria_movimentacao,
    cast(null as string) as subcategoria_evento,
    cast(ids_alvara as string) as ids_alvara,
    cast(qtd_alvaras as bigint) as qtd_alvaras,
    cast(ids_artigo as string) as ids_artigo,
    cast(ds_tipificacao_penal as string) as ds_tipificacao_penal,
    cast(ds_tipificacao_penal_principal as string) as ds_tipificacao_penal_principal,
    cast(qtd_tipificacoes_penais as bigint) as qtd_tipificacoes_penais,
    cast(ids_estabelecimento_externo as string) as ids_estabelecimento_externo,
    cast(ds_estabelecimento_externo as string) as ds_estabelecimento_externo,
    cast(ids_estabelecimento_security as string) as ids_estabelecimento_security,
    cast(ids_estabelecimento_anterior as string) as ids_estabelecimento_anterior,
    cast(ids_tipo_obito as string) as ids_tipo_obito,
    cast(ds_tipo_obito as string) as ds_tipo_obito,
    cast(ids_tipo_saida_temporaria as string) as ids_tipo_saida_temporaria,
    cast(ds_tipo_saida_temporaria as string) as ds_tipo_saida_temporaria,
    cast(dt_retorno_saida_temporaria as timestamp) as dt_retorno_saida_temporaria,
    1 as prioridade_categoria
from gold.sinp_ent_mov_entrada
where id_preso is not null
  and movimentacao_data is not null

union all

select
    cast(id_preso as string) as id_preso,
    cast(id_pessoa as string) as id_pessoa,
    nome_pessoa,
    cast(id_movimentacao as string) as id_movimentacao,
    cast(id_tipomovimentacao as string) as id_tipomovimentacao,
    ds_tipo_mov,
    cast(movimentacao_data as timestamp) as dt_evento,
    cast(id_estabelecimentosecurity as string) as id_estabelecimento,
    observacao,
    categoria_movimentacao,
    cast(null as string) as subcategoria_evento,
    cast(ids_alvara as string) as ids_alvara,
    cast(qtd_alvaras as bigint) as qtd_alvaras,
    cast(ids_artigo as string) as ids_artigo,
    cast(ds_tipificacao_penal as string) as ds_tipificacao_penal,
    cast(ds_tipificacao_penal_principal as string) as ds_tipificacao_penal_principal,
    cast(qtd_tipificacoes_penais as bigint) as qtd_tipificacoes_penais,
    cast(ids_estabelecimento_externo as string) as ids_estabelecimento_externo,
    cast(ds_estabelecimento_externo as string) as ds_estabelecimento_externo,
    cast(ids_estabelecimento_security as string) as ids_estabelecimento_security,
    cast(ids_estabelecimento_anterior as string) as ids_estabelecimento_anterior,
    cast(ids_tipo_obito as string) as ids_tipo_obito,
    cast(ds_tipo_obito as string) as ds_tipo_obito,
    cast(ids_tipo_saida_temporaria as string) as ids_tipo_saida_temporaria,
    cast(ds_tipo_saida_temporaria as string) as ds_tipo_saida_temporaria,
    cast(dt_retorno_saida_temporaria as timestamp) as dt_retorno_saida_temporaria,
    2 as prioridade_categoria
from gold.sinp_ent_mov_saida
where id_preso is not null
  and movimentacao_data is not null
""")

df_eventos_menores = spark.sql("""
select
    cast(id_preso as string) as id_preso,
    cast(id_pessoa as string) as id_pessoa,
    nome_pessoa,
    cast(id_movimentacao as string) as id_movimentacao,
    cast(id_tipomovimentacao as string) as id_tipomovimentacao,
    ds_tipo_mov,
    cast(movimentacao_data as timestamp) as dt_evento,
    cast(id_estabelecimentosecurity as string) as id_estabelecimento,
    observacao,
    categoria_movimentacao,
    cast(subcategoria_saidinha as string) as subcategoria_evento,
    cast(ids_alvara as string) as ids_alvara,
    cast(qtd_alvaras as bigint) as qtd_alvaras,
    cast(ids_artigo as string) as ids_artigo,
    cast(ds_tipificacao_penal as string) as ds_tipificacao_penal,
    cast(ds_tipificacao_penal_principal as string) as ds_tipificacao_penal_principal,
    cast(qtd_tipificacoes_penais as bigint) as qtd_tipificacoes_penais,
    cast(ids_estabelecimento_externo as string) as ids_estabelecimento_externo,
    cast(ds_estabelecimento_externo as string) as ds_estabelecimento_externo,
    cast(ids_estabelecimento_security as string) as ids_estabelecimento_security,
    cast(ids_estabelecimento_anterior as string) as ids_estabelecimento_anterior,
    cast(ids_tipo_obito as string) as ids_tipo_obito,
    cast(ds_tipo_obito as string) as ds_tipo_obito,
    cast(ids_tipo_saida_temporaria as string) as ids_tipo_saida_temporaria,
    cast(ds_tipo_saida_temporaria as string) as ds_tipo_saida_temporaria,
    cast(dt_retorno_saida_temporaria as timestamp) as dt_retorno_saida_temporaria,
    case
        when categoria_movimentacao = 'MOVIMENTACOES_INTERNAS' then 3
        when categoria_movimentacao = 'MOVIMENTACOES_EXTERNAS' then 4
        when categoria_movimentacao = 'MOVIMENTACOES_SAIDINHA' then 5
        else 9
    end as prioridade_categoria
from (
    select * from gold.sinp_ent_mov_interna
    union all
    select * from gold.sinp_ent_mov_externa
    union all
    select * from gold.sinp_ent_mov_saidinha
) x
where id_preso is not null
  and movimentacao_data is not null
""")

df_eventos_todos = (
    df_eventos_macro
    .unionByName(df_eventos_menores)
    .withColumn(
        "ord_tp_evento",
        F.when(F.col("categoria_movimentacao") == "ENTRADA", F.lit(1))
         .when(F.col("categoria_movimentacao") == "SAIDA", F.lit(2))
         .when(F.col("categoria_movimentacao") == "MOVIMENTACOES_INTERNAS", F.lit(3))
         .when(F.col("categoria_movimentacao") == "MOVIMENTACOES_EXTERNAS", F.lit(4))
         .when(F.col("categoria_movimentacao") == "MOVIMENTACOES_SAIDINHA", F.lit(5))
         .otherwise(F.lit(9))
    )
    .repartition("id_preso")
    .sortWithinPartitions("id_preso", "dt_evento", "ord_tp_evento", "prioridade_categoria", "id_movimentacao")
)

# ============================================================
# 2. SCHEMAS
# ============================================================

schema_encarceramento = T.StructType([
    T.StructField("id_encarceramento", T.StringType(), False),
    T.StructField("id_preso", T.StringType(), True),
    T.StructField("id_pessoa", T.StringType(), True),
    T.StructField("nome_pessoa", T.StringType(), True),
    T.StructField("nr_periodo_encarceramento", T.IntegerType(), True),

    T.StructField("id_movimentacao_entrada", T.StringType(), True),
    T.StructField("id_tipomovimentacao_entrada", T.StringType(), True),
    T.StructField("ds_tipo_mov_entrada", T.StringType(), True),
    T.StructField("dt_entrada", T.TimestampType(), True),
    T.StructField("id_estabelecimento_entrada", T.StringType(), True),
    T.StructField("observacao_entrada", T.StringType(), True),
    T.StructField("ids_artigo_entrada", T.StringType(), True),
    T.StructField("ds_tipificacao_penal_entrada", T.StringType(), True),
    T.StructField("ds_tipificacao_penal_principal_entrada", T.StringType(), True),

    T.StructField("id_movimentacao_saida", T.StringType(), True),
    T.StructField("id_tipomovimentacao_saida", T.StringType(), True),
    T.StructField("ds_tipo_mov_saida", T.StringType(), True),
    T.StructField("dt_saida", T.TimestampType(), True),
    T.StructField("id_estabelecimento_saida", T.StringType(), True),
    T.StructField("observacao_saida", T.StringType(), True),
    T.StructField("tp_fechamento", T.StringType(), True),

    T.StructField("st_encarceramento", T.StringType(), True),
    T.StructField("qt_dias_encarceramento", T.IntegerType(), True),

    T.StructField("qtd_mov_internas", T.IntegerType(), True),
    T.StructField("qtd_mov_externas", T.IntegerType(), True),
    T.StructField("qtd_mov_saidinha", T.IntegerType(), True),
    T.StructField("qtd_saida_saidinha", T.IntegerType(), True),
    T.StructField("qtd_retorno_saidinha", T.IntegerType(), True),
    T.StructField("qtd_alvaras_periodo", T.IntegerType(), True),

    T.StructField("dt_primeira_mov_interna", T.TimestampType(), True),
    T.StructField("dt_ultima_mov_interna", T.TimestampType(), True),
    T.StructField("dt_primeira_mov_externa", T.TimestampType(), True),
    T.StructField("dt_ultima_mov_externa", T.TimestampType(), True),
    T.StructField("dt_primeira_saidinha", T.TimestampType(), True),
    T.StructField("dt_ultima_saidinha", T.TimestampType(), True),

    T.StructField("ds_eventos_internos", T.StringType(), True),
    T.StructField("ds_eventos_externos", T.StringType(), True),
    T.StructField("ds_eventos_saidinha", T.StringType(), True),

    T.StructField("score_comportamental", T.DoubleType(), True),
    T.StructField("perfil_comportamental", T.StringType(), True),

    T.StructField("fl_inconsistente", T.StringType(), True)
])

schema_evento_periodo = T.StructType([
    T.StructField("id_encarceramento_evento", T.StringType(), False),
    T.StructField("id_encarceramento", T.StringType(), True),
    T.StructField("id_preso", T.StringType(), True),
    T.StructField("id_pessoa", T.StringType(), True),
    T.StructField("nome_pessoa", T.StringType(), True),
    T.StructField("nr_periodo_encarceramento", T.IntegerType(), True),
    T.StructField("categoria_movimentacao", T.StringType(), True),
    T.StructField("subcategoria_evento", T.StringType(), True),
    T.StructField("id_movimentacao", T.StringType(), True),
    T.StructField("id_tipomovimentacao", T.StringType(), True),
    T.StructField("ds_tipo_mov", T.StringType(), True),
    T.StructField("dt_evento", T.TimestampType(), True),
    T.StructField("id_estabelecimento", T.StringType(), True),
    T.StructField("observacao", T.StringType(), True),
    T.StructField("ids_alvara", T.StringType(), True),
    T.StructField("qtd_alvaras", T.LongType(), True),
    T.StructField("ids_artigo", T.StringType(), True),
    T.StructField("ds_tipificacao_penal", T.StringType(), True),
    T.StructField("ds_tipificacao_penal_principal", T.StringType(), True),
    T.StructField("qtd_tipificacoes_penais", T.LongType(), True),
    T.StructField("ids_estabelecimento_externo", T.StringType(), True),
    T.StructField("ds_estabelecimento_externo", T.StringType(), True),
    T.StructField("ids_estabelecimento_security", T.StringType(), True),
    T.StructField("ids_estabelecimento_anterior", T.StringType(), True),
    T.StructField("ids_tipo_obito", T.StringType(), True),
    T.StructField("ds_tipo_obito", T.StringType(), True),
    T.StructField("ids_tipo_saida_temporaria", T.StringType(), True),
    T.StructField("ds_tipo_saida_temporaria", T.StringType(), True),
    T.StructField("dt_retorno_saida_temporaria", T.TimestampType(), True)
])

schema_inconsistencia = T.StructType([
    T.StructField("id_encarceramento_inconsistencia", T.StringType(), False),
    T.StructField("id_preso", T.StringType(), True),
    T.StructField("id_pessoa", T.StringType(), True),
    T.StructField("nome_pessoa", T.StringType(), True),
    T.StructField("nr_periodo_encarceramento", T.IntegerType(), True),
    T.StructField("id_movimentacao_ref", T.StringType(), True),
    T.StructField("categoria_movimentacao_ref", T.StringType(), True),
    T.StructField("ds_tipo_mov_ref", T.StringType(), True),
    T.StructField("dt_evento_ref", T.TimestampType(), True),
    T.StructField("tp_inconsistencia", T.StringType(), True),
    T.StructField("detalhe_inconsistencia", T.StringType(), True)
])

# ============================================================
# 3. PROCESSAMENTO PURO PYTHON POR PRESO
# ============================================================

def processar_lista_eventos(eventos):
    eventos = sorted(
        eventos,
        key=lambda r: (
            r["dt_evento"],
            r["ord_tp_evento"],
            r["prioridade_categoria"],
            r["id_movimentacao"] if r["id_movimentacao"] is not None else ""
        )
    )

    encarceramentos = []
    eventos_periodo = []
    inconsistencias = []

    aberto = None
    nr_periodo = 0
    hoje = _today_midnight()

    def novo_periodo(r, nr):
        return {
            "id_preso": r["id_preso"],
            "id_pessoa": r["id_pessoa"],
            "nome_pessoa": r["nome_pessoa"],
            "nr_periodo_encarceramento": int(nr),

            "id_movimentacao_entrada": r["id_movimentacao"],
            "id_tipomovimentacao_entrada": r["id_tipomovimentacao"],
            "ds_tipo_mov_entrada": r["ds_tipo_mov"],
            "dt_entrada": _to_python_dt(r["dt_evento"]),
            "id_estabelecimento_entrada": r["id_estabelecimento"],
            "observacao_entrada": r["observacao"],
            "ids_artigo_entrada": r["ids_artigo"],
            "ds_tipificacao_penal_entrada": r["ds_tipificacao_penal"],
            "ds_tipificacao_penal_principal_entrada": r["ds_tipificacao_penal_principal"],

            "id_movimentacao_saida": None,
            "id_tipomovimentacao_saida": None,
            "ds_tipo_mov_saida": None,
            "dt_saida": None,
            "id_estabelecimento_saida": None,
            "observacao_saida": None,
            "tp_fechamento": None,

            "eventos": [],
            "fl_inconsistente": "N"
        }

    def adicionar_evento(periodo, r):
        periodo["eventos"].append({
            "categoria_movimentacao": r["categoria_movimentacao"],
            "subcategoria_evento": r["subcategoria_evento"],
            "id_movimentacao": r["id_movimentacao"],
            "id_tipomovimentacao": r["id_tipomovimentacao"],
            "ds_tipo_mov": r["ds_tipo_mov"],
            "dt_evento": _to_python_dt(r["dt_evento"]),
            "id_estabelecimento": r["id_estabelecimento"],
            "observacao": r["observacao"],
            "ids_alvara": r["ids_alvara"],
            "qtd_alvaras": int(r["qtd_alvaras"]) if r["qtd_alvaras"] is not None else None,
            "ids_artigo": r["ids_artigo"],
            "ds_tipificacao_penal": r["ds_tipificacao_penal"],
            "ds_tipificacao_penal_principal": r["ds_tipificacao_penal_principal"],
            "qtd_tipificacoes_penais": int(r["qtd_tipificacoes_penais"]) if r["qtd_tipificacoes_penais"] is not None else None,
            "ids_estabelecimento_externo": r["ids_estabelecimento_externo"],
            "ds_estabelecimento_externo": r["ds_estabelecimento_externo"],
            "ids_estabelecimento_security": r["ids_estabelecimento_security"],
            "ids_estabelecimento_anterior": r["ids_estabelecimento_anterior"],
            "ids_tipo_obito": r["ids_tipo_obito"],
            "ds_tipo_obito": r["ds_tipo_obito"],
            "ids_tipo_saida_temporaria": r["ids_tipo_saida_temporaria"],
            "ds_tipo_saida_temporaria": r["ds_tipo_saida_temporaria"],
            "dt_retorno_saida_temporaria": _to_python_dt(r["dt_retorno_saida_temporaria"])
        })

    def fechar_periodo(periodo, r_saida=None, dt_fechamento_forcado=None, tp_fechamento="SAIDA"):
        nonlocal encarceramentos, eventos_periodo

        if r_saida is not None:
            periodo["id_movimentacao_saida"] = r_saida["id_movimentacao"]
            periodo["id_tipomovimentacao_saida"] = r_saida["id_tipomovimentacao"]
            periodo["ds_tipo_mov_saida"] = r_saida["ds_tipo_mov"]
            periodo["dt_saida"] = _to_python_dt(r_saida["dt_evento"])
            periodo["id_estabelecimento_saida"] = r_saida["id_estabelecimento"]
            periodo["observacao_saida"] = r_saida["observacao"]
            periodo["tp_fechamento"] = tp_fechamento
        else:
            periodo["dt_saida"] = dt_fechamento_forcado
            periodo["tp_fechamento"] = tp_fechamento

        id_enc = _hash(
            periodo["id_preso"],
            periodo["nr_periodo_encarceramento"],
            periodo["id_movimentacao_entrada"],
            periodo["dt_entrada"]
        )

        eventos = periodo["eventos"]
        ev_interna = [e for e in eventos if e["categoria_movimentacao"] == "MOVIMENTACOES_INTERNAS"]
        ev_externa = [e for e in eventos if e["categoria_movimentacao"] == "MOVIMENTACOES_EXTERNAS"]
        ev_saidinha = [e for e in eventos if e["categoria_movimentacao"] == "MOVIMENTACOES_SAIDINHA"]

        def min_dt(lst):
            return min([e["dt_evento"] for e in lst]) if lst else None

        def max_dt(lst):
            return max([e["dt_evento"] for e in lst]) if lst else None

        def txt_eventos(lst):
            vals = sorted(set([str(e["ds_tipo_mov"]) for e in lst if e["ds_tipo_mov"] is not None]))
            return ",".join(vals) if vals else None

        qtd_saida_saidinha = sum(1 for e in ev_saidinha if e["subcategoria_evento"] == "SAIDA_SAIDINHA")
        qtd_retorno_saidinha = sum(1 for e in ev_saidinha if e["subcategoria_evento"] == "RETORNO_SAIDINHA")
        qtd_alvaras_periodo = sum(int(e["qtd_alvaras"]) for e in eventos if e["qtd_alvaras"] is not None)

        dt_fim = periodo["dt_saida"] if periodo["dt_saida"] is not None else hoje
        qt_dias = int((dt_fim.date() - periodo["dt_entrada"].date()).days)

        score = (
            len(ev_interna) * 1.0
            + len(ev_externa) * 2.0
            + len(ev_saidinha) * 1.5
            + qtd_alvaras_periodo * 2.0
        )

        if score >= 20:
            perfil = "ALTA_DINAMICA"
        elif score >= 8:
            perfil = "MEDIA_DINAMICA"
        else:
            perfil = "BAIXA_DINAMICA"

        encarceramentos.append((
            id_enc,
            periodo["id_preso"],
            periodo["id_pessoa"],
            periodo["nome_pessoa"],
            int(periodo["nr_periodo_encarceramento"]),

            periodo["id_movimentacao_entrada"],
            periodo["id_tipomovimentacao_entrada"],
            periodo["ds_tipo_mov_entrada"],
            periodo["dt_entrada"],
            periodo["id_estabelecimento_entrada"],
            periodo["observacao_entrada"],
            periodo["ids_artigo_entrada"],
            periodo["ds_tipificacao_penal_entrada"],
            periodo["ds_tipificacao_penal_principal_entrada"],

            periodo["id_movimentacao_saida"],
            periodo["id_tipomovimentacao_saida"],
            periodo["ds_tipo_mov_saida"],
            periodo["dt_saida"],
            periodo["id_estabelecimento_saida"],
            periodo["observacao_saida"],
            periodo["tp_fechamento"],

            "FECHADO" if periodo["dt_saida"] is not None else "ABERTO",
            qt_dias,

            len(ev_interna),
            len(ev_externa),
            len(ev_saidinha),
            qtd_saida_saidinha,
            qtd_retorno_saidinha,
            qtd_alvaras_periodo,

            min_dt(ev_interna),
            max_dt(ev_interna),
            min_dt(ev_externa),
            max_dt(ev_externa),
            min_dt(ev_saidinha),
            max_dt(ev_saidinha),

            txt_eventos(ev_interna),
            txt_eventos(ev_externa),
            txt_eventos(ev_saidinha),

            float(score),
            perfil,

            periodo["fl_inconsistente"]
        ))

        for e in eventos:
            eventos_periodo.append((
                _hash(id_enc, e["id_movimentacao"], e["categoria_movimentacao"]),
                id_enc,
                periodo["id_preso"],
                periodo["id_pessoa"],
                periodo["nome_pessoa"],
                int(periodo["nr_periodo_encarceramento"]),
                e["categoria_movimentacao"],
                e["subcategoria_evento"],
                e["id_movimentacao"],
                e["id_tipomovimentacao"],
                e["ds_tipo_mov"],
                e["dt_evento"],
                e["id_estabelecimento"],
                e["observacao"],
                e["ids_alvara"],
                e["qtd_alvaras"],
                e["ids_artigo"],
                e["ds_tipificacao_penal"],
                e["ds_tipificacao_penal_principal"],
                e["qtd_tipificacoes_penais"],
                e["ids_estabelecimento_externo"],
                e["ds_estabelecimento_externo"],
                e["ids_estabelecimento_security"],
                e["ids_estabelecimento_anterior"],
                e["ids_tipo_obito"],
                e["ds_tipo_obito"],
                e["ids_tipo_saida_temporaria"],
                e["ds_tipo_saida_temporaria"],
                e["dt_retorno_saida_temporaria"]
            ))

    for r in eventos:
        cat = r["categoria_movimentacao"]

        if cat == "ENTRADA":
            if aberto is None:
                nr_periodo += 1
                aberto = novo_periodo(r, nr_periodo)
            else:
                dt_forcada = _to_python_dt(r["dt_evento"]) - timedelta(seconds=1)
                fechar_periodo(
                    aberto,
                    r_saida=None,
                    dt_fechamento_forcado=dt_forcada,
                    tp_fechamento="AJUSTE_NOVA_ENTRADA"
                )

                inconsistencias.append((
                    _hash(r["id_preso"], r["id_movimentacao"], "ENTRADA_SEM_SAIDA_ANTERIOR"),
                    r["id_preso"],
                    r["id_pessoa"],
                    r["nome_pessoa"],
                    int(aberto["nr_periodo_encarceramento"]),
                    r["id_movimentacao"],
                    cat,
                    r["ds_tipo_mov"],
                    _to_python_dt(r["dt_evento"]),
                    "ENTRADA_SEM_SAIDA_ANTERIOR",
                    "Periodo anterior fechado artificialmente em nova entrada - 1 segundo"
                ))

                nr_periodo += 1
                aberto = novo_periodo(r, nr_periodo)

        elif cat == "SAIDA":
            if aberto is None:
                inconsistencias.append((
                    _hash(r["id_preso"], r["id_movimentacao"], "SAIDA_SEM_ENTRADA"),
                    r["id_preso"],
                    r["id_pessoa"],
                    r["nome_pessoa"],
                    None,
                    r["id_movimentacao"],
                    cat,
                    r["ds_tipo_mov"],
                    _to_python_dt(r["dt_evento"]),
                    "SAIDA_SEM_ENTRADA",
                    "Saida encontrada sem periodo aberto"
                ))
            else:
                if _to_python_dt(r["dt_evento"]) < aberto["dt_entrada"]:
                    inconsistencias.append((
                        _hash(r["id_preso"], r["id_movimentacao"], "SAIDA_ANTERIOR_ENTRADA"),
                        r["id_preso"],
                        r["id_pessoa"],
                        r["nome_pessoa"],
                        int(aberto["nr_periodo_encarceramento"]),
                        r["id_movimentacao"],
                        cat,
                        r["ds_tipo_mov"],
                        _to_python_dt(r["dt_evento"]),
                        "SAIDA_ANTERIOR_ENTRADA",
                        "Saida com data anterior a entrada do periodo"
                    ))
                else:
                    fechar_periodo(aberto, r_saida=r, tp_fechamento="SAIDA")
                    aberto = None

        else:
            if aberto is None:
                inconsistencias.append((
                    _hash(r["id_preso"], r["id_movimentacao"], "EVENTO_MENOR_FORA_PERIODO"),
                    r["id_preso"],
                    r["id_pessoa"],
                    r["nome_pessoa"],
                    None,
                    r["id_movimentacao"],
                    cat,
                    r["ds_tipo_mov"],
                    _to_python_dt(r["dt_evento"]),
                    "EVENTO_MENOR_FORA_PERIODO",
                    "Evento menor sem encarceramento aberto para agrupamento"
                ))
            else:
                adicionar_evento(aberto, r)

    if aberto is not None:
        fechar_periodo(aberto, r_saida=None, dt_fechamento_forcado=None, tp_fechamento=None)

    return encarceramentos, eventos_periodo, inconsistencias

def iter_grouped_by_preso(rows_iter):
    current_id = None
    bucket = []

    for row in rows_iter:
        d = row.asDict(recursive=True)
        k = d["id_preso"]

        if current_id is None:
            current_id = k

        if k != current_id:
            yield current_id, bucket
            current_id = k
            bucket = [d]
        else:
            bucket.append(d)

    if current_id is not None:
        yield current_id, bucket

def map_encarceramento(rows_iter):
    for _, eventos in iter_grouped_by_preso(rows_iter):
        encs, _, _ = processar_lista_eventos(eventos)
        for x in encs:
            yield x

def map_eventos(rows_iter):
    for _, eventos in iter_grouped_by_preso(rows_iter):
        _, evs, _ = processar_lista_eventos(eventos)
        for x in evs:
            yield x

def map_inconsistencias(rows_iter):
    for _, eventos in iter_grouped_by_preso(rows_iter):
        _, _, incs = processar_lista_eventos(eventos)
        for x in incs:
            yield x

# ============================================================
# 4. EXECUCAO SEM PANDAS
# ============================================================

base_rdd = df_eventos_todos.rdd

df_fat_encarceramento = spark.createDataFrame(
    base_rdd.mapPartitions(map_encarceramento),
    schema=schema_encarceramento
)

df_fat_encarceramento_evento = spark.createDataFrame(
    base_rdd.mapPartitions(map_eventos),
    schema=schema_evento_periodo
)

df_fat_encarceramento_inconsistencia = spark.createDataFrame(
    base_rdd.mapPartitions(map_inconsistencias),
    schema=schema_inconsistencia
)

# ============================================================
# 5. AJUSTES FINAIS
# ============================================================

df_fat_encarceramento = df_fat_encarceramento.dropDuplicates(["id_encarceramento"])
df_fat_encarceramento_evento = df_fat_encarceramento_evento.dropDuplicates(["id_encarceramento_evento"])
df_fat_encarceramento_inconsistencia = df_fat_encarceramento_inconsistencia.dropDuplicates(["id_encarceramento_inconsistencia"])

# ============================================================
# 6. PERSISTENCIA
# ============================================================

tabela = "sinp_fat_encarceramento"

df_fat_encarceramento.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_encarceramento, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_encarceramento")

tabela = "sinp_fat_encarceramento_evento"

df_fat_encarceramento_evento.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_encarceramento_evento, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_encarceramento_evento")

tabela = "sinp_fat_encarceramento_inconsistencia"

df_fat_encarceramento_inconsistencia.write \
    .mode("overwrite") \
    .option("maxRecordsPerFile", 1_000_000) \
    .option("compression", "snappy") \
    .parquet(f"{path}{tabela}")

write_impala_table_partioned(df_fat_encarceramento_inconsistencia, "gold", tabela, f"{path}{tabela}")
enviar_gold_para_postgres(f"gold.{tabela}", "id_encarceramento_inconsistencia")

# ============================================================
# 7. DIAGNOSTICO
# ============================================================

print("fat_encarceramento:", df_fat_encarceramento.count())
print("fat_encarceramento_evento:", df_fat_encarceramento_evento.count())
print("fat_encarceramento_inconsistencia:", df_fat_encarceramento_inconsistencia.count())

spark.sql("""
select
    st_encarceramento,
    tp_fechamento,
    count(*) as qtd
from gold.sinp_fat_encarceramento
group by st_encarceramento, tp_fechamento
order by 1,2
""").show(200, False)

spark.sql("""
select
    perfil_comportamental,
    count(*) as qtd
from gold.sinp_fat_encarceramento
group by perfil_comportamental
order by 1
""").show(200, False)